# 04 — The Validation Gate

**Notebook 4 of the *Developer Guide to Disciplined Trading* series.**

> Prerequisites: [`01-foundations`](./01-foundations-techtrade-and-analysis.ipynb), [`02-morning-scan`](./02-morning-scan.ipynb), [`03-single-position-deep-dive`](./03-single-position-deep-dive.ipynb). You should have a `TradePlan` in hand and know how to read its votes.

> **Soft dependency**: this notebook requires the `[validation]` extra. If you haven't installed it:
> ```
> pip install 'openbb-techtrade[validation]'
> ```
> Without it, `obb.techtrade.validate(...)` raises `TechtradeDependencyError` with a copy-pasteable hint. Most cells below will degrade cleanly with a skip notice.

---

## The question this notebook answers

> *"Notebook 03 showed me one simulated path. The trade made money on that path. **But was the rule actually a good rule, or did I just get lucky on this one historical window?**"*

This is THE most important question in disciplined trading. Notebook 03 produced a single backtest sample. **One sample is anecdote, not evidence.** A rule that scored beautifully on one path can be catastrophically overfit — meaning it found a pattern in the noise of that specific window that won't repeat.

`obb.techtrade.validate(...)` is the gate. It takes a plan, runs the underlying confluence strategy over **walk-forward (WFO) or combinatorial purged cross-validation (CPCV) folds**, and computes:

- **PBO** — Probability of Backtest Overfitting (Bailey/López de Prado). Low is good (< 0.2 = robust; ≥ 0.5 = overfit).
- **DSR** — Deflated Sharpe Ratio. High is good (> 0.95 = robust; < 0.5 = overfit).
- **OOS Sharpe** — aggregated out-of-sample Sharpe ratio across folds.

...then returns a single one-word **verdict** ∈ `{robust, fragile, overfit}` with the precedence `overfit > robust > fragile` (any overfit trigger wins; robust requires all three favorable).

## What Alex takes away

- A complete `ValidationReport` for one plan, including the verdict and the statistical evidence behind it.
- An intuition for what each verdict actually means in practice.
- A side-by-side comparison of the **same plan under WFO vs CPCV** — when they agree vs disagree, and which to trust.
- A demonstration of the soft-dep degradation when `[validation]` isn't installed.

## Wall-clock

**2-15 minutes** per `validate` call depending on the horizon and fold count. The cell that runs validate is the slowest in the entire notebook series.

## 1. Setup probe + the soft-dep check

In [ ]:
import importlib.util

BACKTEST_AVAILABLE = importlib.util.find_spec("openbb_backtest") is not None
print(f"[validation] extra installed (openbb_backtest):  {BACKTEST_AVAILABLE}")
if not BACKTEST_AVAILABLE:
    print("\nMost cells below will SKIP cleanly. Install with:")
    print("    pip install 'openbb-techtrade[validation]'")
    print("\nThis is the same skipif pattern the integration tests use.")

In [ ]:
from openbb import obb

# Pick a ticker. NVDA is liquid + well-known + usually has a tradeable signal.
SYMBOL = "NVDA"
PRESET = "trend_follow"
print(f"Studying validation for {SYMBOL} under preset='{PRESET}'")

## 2. Build a plan to validate

`validate` takes a `TradePlan`. We need one. Re-run notebook 03's `plan()` call to produce one for our ticker.

In [ ]:
plans = obb.techtrade.plan(symbols=[SYMBOL], preset=PRESET, risk=0.01).results
if not plans:
    raise RuntimeError(f"plan() returned no plans for {SYMBOL} — score below entry_threshold today.")
plan = plans[0]
rec = plan.recommendation
print(f"Plan ready: {plan.symbol}  action={rec.action}  score={plan.signal.score:+.4f}  r:r={rec.risk_reward:.2f}")
print(f"plan.validation is currently: {plan.validation!r}")

## 3. Demonstrate the soft-dep degradation

Before the long-running validate, let's confirm what happens if `[validation]` is absent. **This cell runs even when `openbb_backtest` IS installed** — the example script catches `TechtradeDependencyError` either way. When the extra is genuinely missing, the error message has a copy-pasteable install hint.

In [ ]:
from openbb_techtrade.examples import validate_a_plan
from openbb_techtrade.validation.backtest_bridge import TechtradeDependencyError

try:
    result = validate_a_plan.main(symbol=SYMBOL, method="wfo", horizon_years=1)
    if result is None:
        print("validate_a_plan.main() returned None — typical when [validation] is absent.")
    else:
        print(f"validate_a_plan.main() returned a ValidationReport with verdict={result.verdict}")
except TechtradeDependencyError as exc:
    print(f"TechtradeDependencyError caught (this is the soft-dep contract):\n  {exc}")

### What this shows

The `validate_a_plan.py` example **degrades** when the extra is missing instead of crashing. This is the same pattern the integration test `tests/integration/test_tune.py` uses to stay green on bare installs.

If you want to see the raw raise (without the catch), the cell below tries it directly:

In [ ]:
if not BACKTEST_AVAILABLE:
    # The bare-install path: validate raises immediately.
    try:
        obb.techtrade.validate(plan=plan)
    except TechtradeDependencyError as exc:
        print(f"Raised cleanly with the install hint:\n  {exc}")
else:
    print("[validation] is installed; this cell would not trip the dependency error.")
    print("To see the error path, uninstall openbb-backtest temporarily.")

## 4. Walk-Forward validation (WFO)

If `[validation]` is installed, the next cell is the **headline call**. `method="wfo"` runs walk-forward optimization: it splits the price history into rolling train/test windows, re-runs the strategy on each out-of-sample window, and aggregates the OOS metrics.

**`horizon_years=5`** spans roughly 1260 trading days. **`horizon_years=1`** is faster (~250 days) but produces a noisier verdict.

Wall-clock: **~3-10 minutes** for `horizon_years=5`. Use `horizon_years=1` if you just want to feel the surface.

In [ ]:
HORIZON = 5  # change to 1 if you want this cell to finish faster

if BACKTEST_AVAILABLE:
    print(f"Running validate(method='wfo', horizon_years={HORIZON}) on {SYMBOL}... (~3-10 min for horizon=5)")
    wfo_result = obb.techtrade.validate(
        plan=plan,
        method="wfo",
        horizon_years=HORIZON,
    )
    wfo_report = wfo_result.results
    print(f"\nDone. Verdict: {wfo_report.verdict.upper()}")
else:
    wfo_report = None
    print("[validation] not installed; skipping WFO call.")

In [ ]:
if wfo_report is not None:
    print(f"--- ValidationReport for {SYMBOL} (method=wfo, horizon={HORIZON}y) ---\n")
    print(f"  Verdict:                  {wfo_report.verdict.upper()}")
    print(f"  PBO (lower = better):     {wfo_report.pbo:.4f}")
    print(f"  Deflated Sharpe (higher): {wfo_report.deflated_sharpe:.4f}")
    print(f"  OOS Sharpe:               {wfo_report.oos_metrics.sharpe:.4f}")
    print(f"  Min-backtest-length:      {wfo_report.min_backtest_length_years:.2f} years")
    print(f"  Fold count:               {len(wfo_report.folds)}")
    print(f"\nVerdict thresholds in effect (openbb-backtest defaults):")
    for k, v in wfo_report.thresholds.items():
        print(f"    {k:<14} {v}")

### How to read the verdict

The gate's precedence rule (from `openbb_backtest.validation.report`): **`overfit > robust > fragile`**.

| If you got... | What it means | What Alex does |
|---|---|---|
| **`robust`** | PBO < 0.2 AND DSR > 0.95 AND OOS Sharpe > 0. The rule survived out-of-sample with statistical confidence. | Trust the setup. Apply notebook-03 sizing and take it. |
| **`fragile`** | Not enough evidence to call the edge real, but no evidence it's fake. Middle band. | Half-size the position. Watch behavior for the first few trades. |
| **`overfit`** | PBO ≥ 0.5 OR DSR < 0.5 OR OOS Sharpe ≤ 0. At least one trigger condemned it. | **Walk away.** This rule is fitting noise. |

### Why is the threshold so strict?

The PRD §15 ships `pbo_robust=0.2` and `dsr_robust=0.95` deliberately — they're tight. Looser thresholds let more rules pass and more rules fail in production. The strict setting front-loads the disappointment: most candidates fail. **That's the point.** The 1-in-5 rule that DOES pass under strict thresholds is the one Alex should size into.

## 5. Look at the individual folds

The `folds` field is a list of per-fold OOS metrics. Each fold is one walk-forward window. Looking at the distribution across folds tells Alex whether the verdict is **consistent** or driven by one outlier window.

In [ ]:
import pandas as pd

if wfo_report is not None and wfo_report.folds:
    fold_rows = []
    for f in wfo_report.folds:
        fold_rows.append({
            "fold": f.fold,
            "sharpe": round(f.metrics.sharpe, 4),
            "return": round(getattr(f.metrics, 'total_return', 0.0), 4),
            "max_dd": round(getattr(f.metrics, 'max_drawdown', 0.0), 4),
        })
    fold_df = pd.DataFrame(fold_rows)
    print(fold_df.to_string(index=False))
    print(f"\nSharpe stats: mean={fold_df['sharpe'].mean():.3f}  std={fold_df['sharpe'].std():.3f}  min={fold_df['sharpe'].min():.3f}  max={fold_df['sharpe'].max():.3f}")
else:
    print("No fold detail available (run §4 first).")

### What "consistent" looks like

- **Sharpe std < 0.5 × Sharpe mean**: most folds agree. A `robust` verdict here is well-supported.
- **Sharpe std > 1.0 × Sharpe mean**: high fold-to-fold variance. Even a robust verdict here is sensitive to which window was sampled — treat as `fragile` regardless of the label.
- **One fold has Sharpe much higher than the others**: the verdict is being carried by a single lucky window. If you remove that fold mentally, would the verdict still be robust? If no, you're looking at over-influence by an outlier.

## 6. Combinatorial Purged Cross-Validation (CPCV)

WFO splits the history into one rolling sequence. **CPCV** (López de Prado, 2018) instead samples *combinations* of train/test splits to reduce path-dependency. It's slower per fold but the resulting PBO is more reliable.

Run the same plan under `method="cpcv"` and compare the two verdicts. Disagreement is informative: WFO says robust + CPCV says fragile means the rule works on the chronological sequence but doesn't generalize to permuted ones — usually a sign the rule benefits from a specific market regime.

Wall-clock: **~5-15 minutes** (CPCV evaluates more folds).

In [ ]:
if BACKTEST_AVAILABLE:
    print(f"Running validate(method='cpcv', horizon_years={HORIZON}) on {SYMBOL}... (~5-15 min)")
    cpcv_result = obb.techtrade.validate(
        plan=plan,
        method="cpcv",
        horizon_years=HORIZON,
    )
    cpcv_report = cpcv_result.results
    print(f"Done. CPCV verdict: {cpcv_report.verdict.upper()}")
else:
    cpcv_report = None
    print("[validation] not installed; skipping CPCV call.")

In [ ]:
if wfo_report is not None and cpcv_report is not None:
    print(f"--- WFO vs CPCV comparison for {SYMBOL} ---\n")
    print(f"  {'metric':<22} {'WFO':>12} {'CPCV':>12}")
    print(f"  {'-'*22} {'-'*12} {'-'*12}")
    print(f"  {'Verdict':<22} {wfo_report.verdict.upper():>12} {cpcv_report.verdict.upper():>12}")
    print(f"  {'PBO':<22} {wfo_report.pbo:>12.4f} {cpcv_report.pbo:>12.4f}")
    print(f"  {'Deflated Sharpe':<22} {wfo_report.deflated_sharpe:>12.4f} {cpcv_report.deflated_sharpe:>12.4f}")
    print(f"  {'OOS Sharpe':<22} {wfo_report.oos_metrics.sharpe:>12.4f} {cpcv_report.oos_metrics.sharpe:>12.4f}")
    print(f"  {'Folds':<22} {len(wfo_report.folds):>12} {len(cpcv_report.folds):>12}")

    if wfo_report.verdict == cpcv_report.verdict:
        print(f"\n  ✓ Both methods agree: {wfo_report.verdict.upper()}")
    else:
        print(f"\n  ⚠ Methods DISAGREE: WFO={wfo_report.verdict} vs CPCV={cpcv_report.verdict}")
        print("    Treat as the weaker of the two. Alex defaults to the more conservative verdict.")

## 7. Attaching the verdict to the plan

`validate` returns the report; the design contract (PRD §15) is that the report **attaches to `plan.validation`**. The bridge function `validate_plan` (from `openbb_techtrade.validation.backtest_bridge`) returns both the updated plan AND the report. Alex's downstream code can then `if plan.validation is not None and plan.validation.verdict == "robust":`.

Below we use the bridge directly to get the attached plan back.

In [ ]:
if BACKTEST_AVAILABLE:
    import asyncio
    from openbb_techtrade.validation.backtest_bridge import validate_plan

    # Use the already-computed WFO horizon for speed; this is the canonical
    # downstream pattern (verdict attached to the plan, ready for the rest
    # of Alex's pipeline to consume).
    updated_plan, attached_report = asyncio.run(
        validate_plan(plan, method="wfo", horizon_years=HORIZON)
    )
    print(f"updated_plan.validation is now: {attached_report.verdict}")
    print(f"  Same as plan.validation? {updated_plan.validation is attached_report}")
    print(f"  Original plan unchanged?  {plan.validation is None}")

### Why is the original `plan.validation` still None?

Immutable discipline. `validate_plan` returns a **new plan** with the report attached (`plan.model_copy(update={"validation": report})`); the original `plan` is untouched. This means Alex can validate the same plan under multiple methods without state-leak between calls.

## 8. Now what? The decision table

Alex has a verdict. He's NOT going to take a trade on a single ticker just because the verdict says robust. The verdict is **one input** to his decision. Here's the table he wrote on the wall:

In [ ]:
decision_table = pd.DataFrame([
    {"verdict": "robust",  "r:r >= 2.0": "yes", "vote agreement": "3/4 families", "action": "Full size (notebook 03 qty)"},
    {"verdict": "robust",  "r:r >= 2.0": "yes", "vote agreement": "2/4 families", "action": "Half size"},
    {"verdict": "robust",  "r:r >= 2.0": "no",  "vote agreement": "any",         "action": "SKIP — bad math regardless of verdict"},
    {"verdict": "fragile", "r:r >= 2.0": "yes", "vote agreement": "3/4 families", "action": "Quarter size, observe 5 trades"},
    {"verdict": "fragile", "r:r >= 2.0": "yes", "vote agreement": "2/4 families", "action": "SKIP — not enough evidence"},
    {"verdict": "overfit", "r:r >= 2.0": "any", "vote agreement": "any",         "action": "SKIP — fitting noise; don't trade"},
])
decision_table

### Calibrate this for yourself

This table is **Alex's** discipline; it's not the engine's. You'll want a different one. Two principles to keep:

1. **`overfit` is always a skip.** Never an exception. The whole point of running validate is to honor this column.
2. **Position sizing is the lever.** Verdict + R:R + vote agreement should produce a sizing fraction (0.0 - 1.0 × notebook-03 qty), not just a yes/no. "Don't trade" and "full size" are the endpoints of a continuum, not the only options.

## 9. What's next

- **Notebook 05 — Per-Sector Tuning**: `obb.techtrade.tune(segment, ...)`. If validate said `fragile` or `overfit` for a sector, could *better indicator periods* turn it into `robust`? Tuneta proposes new periods, validate gates them, the gate persists only the robust ones to disk.
- **Notebook 06 — Audit and Replay**: load yesterday's actionable Excel + the validation verdicts that came with it, replay what Alex did vs what the engine said. Discipline journal.

## Recommended reading on the math

- **PBO (Probability of Backtest Overfitting)** — Bailey & López de Prado, *The Probability of Backtest Overfitting* (2015). The CSCV (Combinatorially Symmetric Cross-Validation) framework underlying `validate`.
- **DSR (Deflated Sharpe Ratio)** — Bailey & López de Prado, *The Deflated Sharpe Ratio* (2014). How to penalize Sharpe for the number of trials.
- **CPCV** — López de Prado, *Advances in Financial Machine Learning* (2018), Ch. 7. Why combinatorial splits beat walk-forward for path-dependent strategies.

All three are the conceptual foundation of why the gate ships with strict defaults.

---

*End of notebook 04.*